In [9]:
import pandas as pd
from itertools import product
import numpy as np

In [10]:
# search params
# https://crfm.stanford.edu/helm/lite/latest/#/leaderboard Jan 2, 2024
llm_stanford = [
    "GPT-4",
    "GPT-4-Turbo", 
    "Palmyra-X-V3",
    "Palmyra-X", #base
    "Palmyra", #base
    "PaLM-2 unicorn", #unicorn
    "PaLM 2", #base
    "PaLM", #base
    "Palmyra-X-V2",
    "Palmyra X", #base
    "Yi 6B",
    "Mixtral 8x7B",
    "Mixtral", #base
    "Claude v1.3",
    "Claude", #base
    "PaLM-2 bison", #bison
    "Claude 2.0",
    "Llama 2",
    "Llama", #base
    "text-davinci-003",
    "text-davinci", #base
    "Claude 2.1",
    "Claude", #base
    "Claude Instant 1.2",
    "text-davinci-002",
]

# https://huggingface.co/spaces/lmsys/chatbot-arena-leaderboard Jan 2, 2024
llm_arena_elo = [
    "GPT-4-Turbo",
    "GPT-4-0314",
    "GPT-4-0613",
    "Claude 1",
    "Claude 2.0",
    "Mixtral-8x7b-Instruc",
    "Mixtral-8x7b", #base
    "Claude 2.1",
    "GPT-3.5-Turbo-0613",
    "GPT-3.5-Turbo", #base
    "GPT-3.5", #base
    "GPT-3", #base
    "GPT", #base
    "Gemini Pro",
    "Gemini", #base
    "Claude-Instant-1",
    "Claude-Instant",
    "Tulū-2-DPO-70B",
    "Tulū 2",  #base
    "Yi-34B-Chat",
    "Yi 34B", #base
    "GPT-3.5-Turbo-0314",
    "WizardLM-70B-v1.0",
    "WizardLM",#base
    "Vicuna-33B"
    "Vicuna",
]

llms = llm_stanford + llm_arena_elo
llms = list(set(llms))

primary_keywords = [
    "Generative Artificial Intelligence",
    "Generative AI",
    "GenAI",
    "Gen AI",
    "LLM",
    "LM",
    "large language model",
    "language model",
    "small language model",
    "compact language model",
    "foundation model",
    
] + llms

secondary_keywords = [
    "overreliance",
    "over-reliance",
    "misinformation",
    "accessibility",
    "privacy",
    "environmental",
    "explainability",
    "trustworthy",
    "responsible",
]

third_level_keywords1 = ["mitigation", "ethics", "societal", "social", "ethical"]

third_level_keywords2 = ["education"]

all_combinations = list(product(primary_keywords, secondary_keywords, third_level_keywords2))
all_combinations = [tuple(combination) for combination in all_combinations]

In [11]:
# reformat file for further processing
# add an extra column called "formattedSearchString" to the dataframe, using data from the "searchString" column
search_result_dblp = pd.read_csv("data/raw/edu/search-results-dblp.csv")
search_result_sch = pd.read_csv("data/raw/edu/search-results-sch.csv")
search_result_web_of_science = pd.read_csv("data/raw/edu/search-results-web-of-science.csv")

search_result_dblp["FormattedSearchString"] = search_result_dblp["SearchString"].apply(
    lambda x: [keyword for keyword in x.replace('"$', '').replace('" ', '').split('"') if keyword]
)

search_result_sch["FormattedSearchString"] = search_result_sch["SearchString"].apply(
    lambda x: x.replace('" + "', '|').replace('"', '').split('|')
)

search_result_web_of_science["FormattedSearchString"] = search_result_web_of_science["SearchString"].apply(
    lambda x: x.replace('" "', '|').replace('"', '').split('|')
)

search_results_raw = pd.concat([search_result_dblp, search_result_sch, search_result_web_of_science])


In [12]:
# group by "formattedSearchString" and count the number of rows in each group
# Count occurrences of each combination in the search results
combination_counts = {}
for combination in all_combinations:
    count = search_results_raw['FormattedSearchString'].apply(
        lambda keywords: all(keyword in keywords for keyword in combination)
    ).sum()
    combination_counts[combination] = count

# Convert to DataFrame
report_df = pd.DataFrame(list(combination_counts.items()), columns=['Combination', 'Total_Search_Result_Count'])
# Sorting the DataFrame by count
report_df.sort_values(by='Total_Search_Result_Count', ascending=False, inplace=True)


# Define function to categorize result group
def categorize_group(count):
    if count > 15:
        return 'high'
    elif count > 0:
        return 'medium'
    else:
        return 'low'

# Apply the function to categorize each combination based on its count
report_df['Result_Group'] = report_df['Total_Search_Result_Count'].apply(categorize_group)
report_df.reset_index(drop=True, inplace=True)
report_df.to_csv("data/validation/edu-keywords-rank.csv", index=False)

In [13]:
# Calculate the percentage of data tagged as 'high', 'medium', 'low'
result_group_counts = report_df['Result_Group'].value_counts(normalize=True) * 100

# Convert the series to a dataframe for better readability
result_group_percentages_df = result_group_counts.reset_index()
result_group_percentages_df.columns = ['Result_Group', 'Percentage']

result_group_percentages_df

,Result_Group,Percentage
0,low,75.438596
1,medium,19.493177
2,high,5.068226


In [14]:
# Calculate the actual number of data points tagged as 'high', 'medium', 'low'
result_group_counts_absolute = report_df['Result_Group'].value_counts()

result_group_counts_absolute_df = result_group_counts_absolute.reset_index()
result_group_counts_absolute_df.columns = ['Result_Group', 'Count']

result_group_counts_absolute_df

,Result_Group,Count
0,low,387
1,medium,100
2,high,26


In [15]:
# pick 3 random combinations from each group with a seed
np.random.seed(42)
report_grouped_df = report_df.groupby('Result_Group').apply(lambda x: x.sample(n=3))
report_grouped_df


/var/folders/c3/lgvf1sgs6gjf4_898lm44qt00000gn/T/ipykernel_5008/214580742.py:3: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  report_grouped_df = report_df.groupby('Result_Group').apply(lambda x: x.sample(n=3))


Combination  \
Result_Group                                                        
high         8     (large language model, responsible, education)   
             16                         (LLM, privacy, education)   
             0                   (PaLM, environmental, education)   
low          434           (Palmyra-X, misinformation, education)   
             475           (text-davinci-003, privacy, education)   
             498  (GPT-3.5-Turbo-0314, misinformation, education)   
medium       43        (Generative AI, misinformation, education)   
             100     (text-davinci-003, accessibility, education)   
             69           (Generative AI, trustworthy, education)   

                  Total_Search_Result_Count Result_Group  
Result_Group                                              
high         8                           33         high  
             16                          25         high  
             0                           83         high  
low          434                          0          low  
             475                          0          low  
             498                          0          low  
medium       43                           6       medium  
             100                          1       medium  
             69                           3       medium

In [22]:
import re
def search_papers(df, search_terms):
    results = pd.DataFrame()  # Empty DataFrame to store results
    for terms in search_terms:
        # Join terms into a single regex pattern
        pattern = '.*'.join(map(re.escape, terms))
        # Compile the regex pattern for case-insensitive search
        regex = re.compile(pattern, re.IGNORECASE)
        # Match the pattern in the 'SearchString' column
        matches = df[df['SearchString'].str.contains(regex, na=False)]
        # Append matches to the results DataFrame
        results = pd.concat([results, matches], ignore_index=True)
    return results

# list all papers in master.csv that are being searched using the keywords (edu)
datasets = [
    'data/raw/edu/search-results-dblp.csv',
    'data/raw/edu/search-results-sch.csv',
    'data/raw/edu/search-results-web-of-science.csv'
]

rand_10_keywords_edu = report_grouped_df['Combination'].tolist()

edu_validation_df = pd.DataFrame()
for dataset in datasets:
    df = pd.read_csv(dataset)
    search_results = search_papers(df, rand_10_keywords_edu)
    edu_validation_df = pd.concat([edu_validation_df, search_results], ignore_index=True)

edu_validation_df.to_csv('data/validation/edu-validation.csv', index=False)
edu_validation_df

,PaperTitle,ID,SearchString,SearchedFrom
0,Scientists' Perspectives on the Potential for ...,DOI:10.48550/arXiv.2304.01420,"""large language model"" + ""responsible"" + ""educ...",Semantic Scholar
1,Performance of ChatGPT on the US Fundamentals ...,DOI:10.48550/arXiv.2304.12198,"""large language model"" + ""responsible"" + ""educ...",Semantic Scholar
2,Application of ChatGPT in information literacy...,DOI:10.15291/pubmet.4273,"""large language model"" + ""responsible"" + ""educ...",Semantic Scholar
3,Accuracy of a Large Language Model as a new to...,DOI:10.1080/08164622.2023.2288174,"""large language model"" + ""responsible"" + ""educ...",Semantic Scholar
4,"Vox Populi, Vox ChatGPT: Large Language Models...",DOI:10.48550/arXiv.2311.06207,"""large language model"" + ""responsible"" + ""educ...",Semantic Scholar
...,...,...,...,...
186,Identifying the local factors of resilience du...,DOI:10.1007/s13280-019-01241-7,"""Palmyra"" ""environmental"" ""education""",Web of Science
187,Friend or foe? Exploring the implications of l...,DOI:10.1007/s00146-023-01791-1,"""Generative AI"" ""misinformation"" ""education""",Web of Science
188,Large Language Models in Medical Education: Op...,DOI:10.2196/48291,"""Generative AI"" ""misinformation"" ""education""",Web of Science
189,?So what if ChatGPT wrote it?? Multidisciplina...,DOI:10.1016/j.ijinfomgt.2023.102642,"""Generative AI"" ""misinformation"" ""education""",Web of Science
